# HIPAA-oriented medical RAG with LangChain and TealTiger

This notebook uses synthetic medical FAQs and a deterministic mock model. It demonstrates defense in depth for a RAG pipeline: input PII scanning, retrieval-result scanning before context injection, output scanning, a per-session budget, and JSON audit evidence. No API key or patient data is required.

## 1. Install dependencies

The notebook intentionally uses an in-memory lexical retriever so it can run without FAISS, Chroma, embeddings, or external services.

In [ ]:
%pip install -q "langchain-core>=1.0.0" "langchain-tealtiger>=0.1.0"

In [ ]:
import json
import re
from typing import Any

from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda
from tealtiger.guardrails import PIIDetectionGuardrail

input_guard = PIIDetectionGuardrail({"action": "block"})
output_guard = PIIDetectionGuardrail({"action": "redact"})
audit_log: list[dict[str, Any]] = []
session = {"cost_usd": 0.0, "budget_usd": 0.03, "session_id": "synthetic-session-001"}

documents = [
    Document(page_content="Adults with mild dehydration should follow the care team's oral rehydration plan.", metadata={"source": "synthetic_faq", "topic": "hydration"}),
    Document(page_content="Seek urgent medical care for chest pain, severe breathing difficulty, or sudden weakness.", metadata={"source": "synthetic_faq", "topic": "urgent-care"}),
    Document(page_content="Medication questions should be reviewed with a licensed clinician or pharmacist.", metadata={"source": "synthetic_faq", "topic": "medication"}),
]

def record(stage: str, action: str, **details: Any) -> None:
    audit_log.append({"session_id": session["session_id"], "stage": stage, "action": action, **details})

async def contains_pii(text: str) -> bool:
    result = await input_guard.evaluate(text)
    return not result.passed

async def redact(text: str) -> str:
    result = await output_guard.evaluate(text)
    return result.metadata.get("redacted_text", text)

## 2. Governance-aware RAG stages

The stages are explicit so a reviewer can see where a control applies. Retrieved documents are treated as untrusted context and are scanned before they reach the model.

In [ ]:
async def protect_input(state: dict[str, Any]) -> dict[str, Any]:
    query = state["query"]
    if await contains_pii(query):
        record("input", "blocked_pii")
        raise ValueError("Input blocked: remove patient identifiers before retrieval.")
    record("input", "allowed")
    return state

async def retrieve(state: dict[str, Any]) -> dict[str, Any]:
    terms = set(re.findall(r"[a-z]+", state["query"].lower()))
    ranked = sorted(documents, key=lambda doc: len(terms & set(doc.page_content.lower().split())), reverse=True)
    safe_docs = []
    for doc in ranked[:2]:
        if not await contains_pii(doc.page_content):
            safe_docs.append(doc)
    record("retrieval", "completed", candidates=len(ranked), safe_documents=len(safe_docs))
    return {**state, "docs": safe_docs}

def generate(state: dict[str, Any]) -> dict[str, Any]:
    if session["cost_usd"] + 0.01 > session["budget_usd"]:
        record("budget", "blocked")
        raise ValueError("Session budget exceeded.")
    session["cost_usd"] += 0.01
    context = " ".join(doc.page_content for doc in state["docs"])
    answer = f"Synthetic answer based on approved context: {context}"
    record("generation", "completed", estimated_cost_usd=0.01)
    return {**state, "answer": answer}

async def protect_output(state: dict[str, Any]) -> dict[str, Any]:
    answer = state["answer"]
    safe_answer = await redact(answer)
    record("output", "redacted" if safe_answer != answer else "allowed")
    return {**state, "answer": safe_answer}

rag = (
    RunnableLambda(protect_input)
    | RunnableLambda(retrieve)
    | RunnableLambda(generate)
    | RunnableLambda(protect_output)
)

## 3. Run safe and unsafe cases

In [ ]:
safe_result = await rag.ainvoke({"query": "What should I know about hydration?"})
assert safe_result["answer"]
assert session["cost_usd"] == 0.01

try:
    await rag.ainvoke({"query": "What should I know? Patient SSN is 123-45-6789."})
except ValueError as exc:
    assert "Input blocked" in str(exc)
else:
    raise AssertionError("PII input was not blocked")

print(safe_result["answer"])
print("Audit events:", len(audit_log))

## 4. Verify output protection and audit export

The following isolated check simulates an unsafe model output without invoking a real model.

In [ ]:
unsafe = await protect_output({"answer": "The record number is 123-45-6789."})
assert "123-45-6789" not in unsafe["answer"]
assert any(event["action"] == "redacted" for event in audit_log)

audit_json = json.dumps(audit_log, indent=2)
with open("hipaa_medical_rag_audit.json", "w", encoding="utf-8") as file:
    file.write(audit_json)

print(audit_json)

## Production notes

- Replace the lexical retriever with FAISS, Chroma, or a managed vector store after validating its data-handling controls.
- Replace the mock generator with an approved model endpoint; keep the input, retrieval, output, budget, and audit stages around it.
- Use synthetic or de-identified data in development. This example is an engineering pattern, not medical advice or a HIPAA certification.